**Aim:** Preprocessing scRNA-seq datasets

**Tasks:**
- [X] Load `adata` objects and merge them into one big dataset

In [1]:
import os, sys
import pickle
import numpy as np 
import pandas as pd
from glob import glob
import anndata as ad
import scanpy as sc
import scanpy.external as sce
import scez 

import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

scanpy==1.9.6 anndata==0.10.7 umap==0.5.5 numpy==1.26.4 scipy==1.13.0 pandas==1.5.3 scikit-learn==1.4.2 statsmodels==0.14.2 igraph==0.11.5 pynndescent==0.5.12


In [3]:
from utils import *

In [4]:
# mkdir -pv figures

### Load `cellranger` results and label samples

<!-- https://www.kallistobus.tools/tutorials/kb_aggregate/python/kb_aggregating_count_matrices.html -->



In [ ]:
%%time 
rawdata = {}
for file in sorted(glob('counts/*/outs/raw_feature_bc_matrix.h5')):
    sample = file.split('/')[1].replace('-','')
    print (sample, file)
    
    adata = sc.read_10x_h5(file)#, gex_only=False)
    
    rawdata[sample] = adata
    del adata
    print ('_'*50)

___

In [ ]:
%%time 
data = {}
for file in sorted(glob('counts/*/outs/filtered_feature_bc_matrix.h5')):
    sample = file.split('/')[1].replace('-','')
    print (sample, file)
    
    adata = sc.read_10x_h5(file)#, gex_only=False)
    
    data[sample] = adata
    del adata
    print ('_'*50)


In [ ]:
!mkdir -pv preprocessing

In [93]:
name = 'preprocessing/adatas-filtered_feature_bc.pkl'

save_obj(data, name)

___

In [95]:
name = 'preprocessing/adatas-raw_feature_bc.pkl'

save_obj(rawdata, name)

In [96]:
ls -s --block-size=M preprocessing/*.pkl

 528M preprocessing/adatas-filtered_feature_bc.pkl
1305M preprocessing/adatas-raw_feature_bc.pkl
 471M preprocessing/adatas-raw.pkl


___


In [35]:
name = 'preprocessing/adatas-raw_feature_bc.pkl'
rawdata = load_obj(name)

In [36]:
name = 'preprocessing/adatas-filtered_feature_bc.pkl'
data = load_obj(name)

In [37]:
new_keys = [
    'MDMA_male_1',
    'MDMA_acute_female_1',
    'MDMA_acute_female_2',
    'MDMA_acute_female_3',
    'MDMA_acute_male_1',
    'MDMA_acute_male_2',
    'MDMA_acute_male_3',

    'MDMA_male_2',
    'MDMA_male_3',
    
    'Meth_male_1',
    'Meth_male_2',
    'Meth_male_3',
    
    'Saline_male_1',
    'Saline_male_2',
    'Saline_male_3'
]

In [38]:
def rename(d, new_keys):
    """
    https://stackoverflow.com/questions/16475384/rename-a-dictionary-key
    """
    new_dict = {}
    for key, new_key in zip(d.keys(), new_keys):
        new_dict[new_key] = d[key]
        print(f'{new_key} <- {key}')
        
    return new_dict

In [39]:
rawdata = rename(rawdata,new_keys)

MDMA_male_1 <- MDMAMale
MDMA_acute_female_1 <- MDMA_acute_female_011221
MDMA_acute_female_2 <- MDMA_acute_female_100121
MDMA_acute_female_3 <- MDMA_acute_female_180921
MDMA_acute_male_1 <- MDMA_acute_male
MDMA_acute_male_2 <- MDMA_acute_male_021221_1
MDMA_acute_male_3 <- MDMA_acute_male_021221_2
MDMA_male_2 <- MDMA_male_093021
MDMA_male_3 <- MDMA_male_100121
Meth_male_1 <- Meth
Meth_male_2 <- Meth_093021
Meth_male_3 <- SalineMale
Saline_male_1 <- Saline_093021
Saline_male_2 <- Saline_202110


___

In [40]:
data = rename(data,new_keys)

MDMA_male_1 <- MDMAMale
MDMA_acute_female_1 <- MDMA_acute_female_011221
MDMA_acute_female_2 <- MDMA_acute_female_100121
MDMA_acute_female_3 <- MDMA_acute_female_180921
MDMA_acute_male_1 <- MDMA_acute_male
MDMA_acute_male_2 <- MDMA_acute_male_021221_1
MDMA_acute_male_3 <- MDMA_acute_male_021221_2
MDMA_male_2 <- MDMA_male_093021
MDMA_male_3 <- MDMA_male_100121
Meth_male_1 <- Meth
Meth_male_2 <- Meth_093021
Meth_male_3 <- Meth_100121
Saline_male_1 <- SalineMale
Saline_male_2 <- Saline_093021
Saline_male_3 <- Saline_202110


### Merge datasets

In [70]:
def MergeDatasets(data_dict):

    ADATAs = []

    for sample in data_dict:

        print (sample)
        exp = data_dict[sample].copy()
        exp.var_names_make_unique()
        
        exp.obs['batch'] = sample
        ADATAs.append(exp)

    adata = ad.concat(ADATAs)

    adata.obs['cond'] = adata.obs['batch'].str[:-2]

    for c in set(adata.obs['cond']): print(c)

    adata.obs['group'] = adata.obs.cond.str.split('_').str[0]

    adata.obs.group = pd.Categorical(adata.obs.group)

    # adata.obs_names_make_unique()
    adata.obs.index = [f'{idx}_{btc}' for idx, btc in zip(adata.obs.index.to_list(),adata.obs.batch.to_list())]
    
    adata.var = ADATAs[0].var
    
    return adata

___

#### Merge `filtered_feature_bc`

In [71]:
adata_filtered_feature_bc = MergeDatasets(data)

MDMA_male_1
MDMA_acute_female_1
MDMA_acute_female_2
MDMA_acute_female_3
MDMA_acute_male_1
MDMA_acute_male_2


/data_gilbert/home/aarab/anaconda3/envs/mamba/envs/scez/lib/python3.9/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/data_gilbert/home/aarab/anaconda3/envs/mamba/envs/scez/lib/python3.9/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/data_gilbert/home/aarab/anaconda3/envs/mamba/envs/scez/lib/python3.9/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/data_gilbert/home/aarab/anaconda3/envs/mamba/envs/scez/lib/python3.9/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/data_gi

MDMA_acute_male_3
MDMA_male_2
MDMA_male_3
Meth_male_1
Meth_male_2
Meth_male_3


/data_gilbert/home/aarab/anaconda3/envs/mamba/envs/scez/lib/python3.9/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/data_gilbert/home/aarab/anaconda3/envs/mamba/envs/scez/lib/python3.9/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/data_gilbert/home/aarab/anaconda3/envs/mamba/envs/scez/lib/python3.9/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Saline_male_1
Saline_male_2
Saline_male_3


/data_gilbert/home/aarab/anaconda3/envs/mamba/envs/scez/lib/python3.9/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


MDMA_acute_male
Saline_male
Meth_male
MDMA_male
MDMA_acute_female


/data_gilbert/home/aarab/anaconda3/envs/mamba/envs/scez/lib/python3.9/site-packages/anndata/_core/anndata.py:1818: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [64]:
results_file = 'preprocessing/adata-merged-filtered_feature_bc-v3.h5ad.gz'

adata_filtered_feature_bc.write(results_file,compression='gzip')

#### Merge `raw_feature_bc`

In [72]:
adata_raw_feature_bc = MergeDatasets(rawdata)

MDMA_male_1
MDMA_acute_female_1
MDMA_acute_female_2


/data_gilbert/home/aarab/anaconda3/envs/mamba/envs/scez/lib/python3.9/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/data_gilbert/home/aarab/anaconda3/envs/mamba/envs/scez/lib/python3.9/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/data_gilbert/home/aarab/anaconda3/envs/mamba/envs/scez/lib/python3.9/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/data_gilbert/home/aarab/anaconda3/envs/mamba/envs/scez/lib/python3.9/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


MDMA_acute_female_3
MDMA_acute_male_1
MDMA_acute_male_2


/data_gilbert/home/aarab/anaconda3/envs/mamba/envs/scez/lib/python3.9/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/data_gilbert/home/aarab/anaconda3/envs/mamba/envs/scez/lib/python3.9/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


MDMA_acute_male_3
MDMA_male_2


/data_gilbert/home/aarab/anaconda3/envs/mamba/envs/scez/lib/python3.9/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/data_gilbert/home/aarab/anaconda3/envs/mamba/envs/scez/lib/python3.9/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/data_gilbert/home/aarab/anaconda3/envs/mamba/envs/scez/lib/python3.9/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


MDMA_male_3
Meth_male_1
Meth_male_2


/data_gilbert/home/aarab/anaconda3/envs/mamba/envs/scez/lib/python3.9/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/data_gilbert/home/aarab/anaconda3/envs/mamba/envs/scez/lib/python3.9/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/data_gilbert/home/aarab/anaconda3/envs/mamba/envs/scez/lib/python3.9/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/data_gilbert/home/aarab/anaconda3/envs/mamba/envs/scez/lib/python3.9/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Meth_male_3
Saline_male_1
Saline_male_2


/data_gilbert/home/aarab/anaconda3/envs/mamba/envs/scez/lib/python3.9/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/data_gilbert/home/aarab/anaconda3/envs/mamba/envs/scez/lib/python3.9/site-packages/anndata/_core/anndata.py:1818: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


MDMA_acute_male
Saline_male
Meth_male
MDMA_male
MDMA_acute_female


In [73]:
adata_raw_feature_bc

AnnData object with n_obs × n_vars = 13433055 × 32285
    obs: 'batch', 'cond', 'group'
    var: 'gene_ids', 'feature_types', 'genome'

In [74]:
results_file = 'preprocessing/adata-merged-raw_feature_bc-v3.h5ad.gz'
adata_raw_feature_bc.write(results_file,compression='gzip')

___

#### 

In [76]:
!ls -lh preprocessing/adata-merged*

-rw-rw-r--. 1 aarab aarab 133M May 20  2024 preprocessing/adata-merged-filtered_feature_bc-v3.h5ad.gz
-rw-rw-r--. 1 aarab aarab 971M May 20  2024 preprocessing/adata-merged-raw_feature_bc-v3.h5ad.gz


# 
<!-- Wed Apr 27 15:50:17 PDT 2022 -->

In [82]:
!date

Sun May 19 02:34:26 PDT 2024


In [83]:
%reload_ext watermark
%watermark
%watermark --iversions

Last updated: 2024-05-19T02:34:26.223459-07:00

Python implementation: CPython
Python version       : 3.9.19
IPython version      : 8.18.1

Compiler    : GCC 12.3.0
OS          : Linux
Release     : 3.10.0-957.27.2.el7.x86_64
Machine     : x86_64
Processor   : x86_64
CPU cores   : 64
Architecture: 64bit

pandas    : 1.5.3
anndata   : 0.10.7
matplotlib: 3.6.3
scanpy    : 1.9.6
numpy     : 1.26.4
scez      : 0.0.5
seaborn   : 0.12.2
sys       : 3.9.19 | packaged by conda-forge | (main, Mar 20 2024, 12:50:21) 
[GCC 12.3.0]

